# Spaceship Titanic with CatcatboostBoost

# 环境准备

In [ ]:
import catboost as cat
import pandas as pd
import numpy as np
import sklearn.model_selection
import seaborn as sns
import matplotlib.pyplot as plt

import warnings 
warnings.filterwarnings('ignore')
cat.__version__

# 加载数据

In [ ]:
dataset_df = pd.read_csv('/kaggle/input/spaceship-titanic/train.csv', engine='pyarrow')
print("Full train dataset shape is {}".format(dataset_df.shape))

数据总共14列8693行，查看返回前五行

In [ ]:
dataset_df.head(3)

共12特征列. 用模型预测是否运输成功：`Transported`

# EDA

In [ ]:
dataset_df.describe()

In [ ]:
dataset_df.info()

查看标签列 Transported 分布情况 



In [ ]:
plot_df = dataset_df.Transported.value_counts()
plot_df.plot(kind="bar")

查看数值特征分布情况

In [ ]:
fig, ax = plt.subplots(5,1,  figsize=(10, 10))
plt.subplots_adjust(top = 2)

sns.histplot(dataset_df['Age'], color='b', bins=30, ax=ax[0]).set_ylabel("Number of people");
sns.histplot(dataset_df['FoodCourt'], color='b', bins=30, ax=ax[1]).set_ylabel("Number of people");
sns.histplot(dataset_df['ShoppingMall'], color='b', bins=30, ax=ax[2]).set_ylabel("Number of people");
sns.histplot(dataset_df['Spa'], color='b', bins=30, ax=ax[3]).set_ylabel("Number of people");
sns.histplot(dataset_df['VRDeck'], color='b', bins=30, ax=ax[4]).set_ylabel("Number of people");

# ETL

清理 `PassengerId` `Name` 等不需要特征

In [ ]:
dataset_df.drop(['PassengerId', 'Name'], axis=1, inplace=True)
dataset_df.head(5)

检查空值情况

In [ ]:
dataset_df.isnull().sum().sort_values(ascending=False)

指定 `Transported` 为标签列，并boolean转换为int

In [ ]:
dataset_df["Transported"] = dataset_df['Transported'].replace({True: 1, False: 0})
dataset_df['VIP'] = dataset_df['VIP'].replace({True: 1, False: 0})
dataset_df['CryoSleep'] = dataset_df['CryoSleep'].replace({True: 1, False: 0})

原列`Cabin` 形为 `Deck/Cabin_num/Side`. 拆分列 3 列 `Deck`, `Cabin_num` and `Side`,这样能够更好训练数据

In [ ]:
dataset_df[["Deck", "Cabin_num", "Side"]] = dataset_df["Cabin"].str.split("/", expand=True)
dataset_df.drop('Cabin', axis=1, inplace=True)

类别特征空值处理，统一为nan，类别特征转换为特定字符null

In [ ]:
cf = ['HomePlanet','Destination','Deck','Side']
dataset_df.replace({None: np.nan}, inplace=True)
dataset_df[cf]=dataset_df[cf].replace({np.nan: 'null'})
dataset_df.isnull().sum().sort_values(ascending=False)

# 数据集
训练集、验证集、测试集

In [ ]:
X, X_test, y, y_test = sklearn.model_selection.train_test_split(
    dataset_df.drop(['Transported'], axis=1),
    dataset_df['Transported'], 
    test_size=0.2,
    shuffle=True,
    random_state=42
)
X_train, X_valid, y_train, y_valid = sklearn.model_selection.train_test_split(X, y, train_size=0.6, shuffle=True, random_state=42)
# data = cat.Pool(dataset_df.drop(['Transported'], axis=1), dataset_df['Transported'], cat_features=cf)
data_train = cat.Pool(X_train, y_train, cat_features=cf)

# 模型构建

# 配置模型

目标函数
* binary:logistic和 'objective': 'reg:logistic'的输出是一样的,都是预测的概率
* binary:logitraw是输出的得分，用sigmoid（）函数处理后就和上述两个概率值一致

* 二类评价函数：'objective': 'binary:logistic','eval_metric': ['AUC','error','logloss'],
* 分类评价函数：'objective': 'multi:softmax','num_class': 3, 'eval_metric': ['auc','merror','mlogloss'],
* 回归评价函数：'objective': 'reg:linear', 'eval_metric': ['rmse','logloss','mae'],



In [ ]:
params = {
    "objective": "CrossEntropy",
    'bootstrap_type': 'Poisson',
    'nan_mode': 'Min',
    'eval_metric':True,
    'use_best_model':True,
    'max_ctr_complexity':4,
    'one_hot_max_size': 2,
    "eval_metric": "AUC",
    "random_seed": 42,
    "task_type": "GPU",
    "devices": "0:1"
}

* 交叉验证方法有k折交叉验证（k-fold cross validation）和留一交叉验证（leave-one-out cross validation）。 
* 在k折交叉验证中，数据集被平均分成k份，其中k-1份用作训练集，剩下的1份用作验证集
* 解决数据小情况，还有快速找出最佳迭代即树颗数

In [ ]:
# 实现交叉验证，初超参调优快速找出最佳树棵数等
cv_df = cat.cv(
    params=params, 
    iterations=300,
    nfold=5, 
    verbose=100,
    early_stopping_rounds=10,
    dtrain=data_train,
    as_pandas=True
)

# 训练模型

指定训练参数、boost数量（残差迭代次数）、评价数据集、早停过拟合等

In [ ]:
model = cat.CatBoostClassifier(**params, iterations=5, used_ram_limit='28GB', thread_count=30, gpu_ram_part=0.98, gpu_cat_features_storage='GpuRam')

In [ ]:
import numpy as np
import optuna
from optuna.integration import CatBoostPruningCallback
import catboost as cat
from sklearn.metrics import accuracy_score

def objective(trial: optuna.Trial) -> float:

    param = {
        #"objective": trial.suggest_categorical("objective", ["Logloss", "CrossEntropy"]),
        "objective": trial.suggest_categorical("objective", ["Logloss"]),
        "depth": trial.suggest_int("depth", 3, 12),
        "learning_rate": trial.suggest_loguniform("learning_rate", 1e-3, 1e0), 
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.01, 1.0, log=True),
        "l2_leaf_reg": trial.suggest_loguniform("l2_leaf_reg", 1e-3, 1e1),
        "max_ctr_complexity": trial.suggest_int("max_ctr_complexity", 1, 4), 
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 3, 30), 
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 0.6, 1.6), 
        "boosting_type": trial.suggest_categorical("boosting_type", ["Ordered", "Plain"]),
        "bootstrap_type": trial.suggest_categorical(
            "bootstrap_type", ["Bayesian", "Bernoulli", "MVS"]
            #"bootstrap_type", ['Poisson']
        ),
        "eval_metric": "Accuracy",
        #"eval_metric": "Logloss",
    }
    
    if param["bootstrap_type"] == "Bayesian":
        param["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0, 10)
    elif param["bootstrap_type"] == "Bernoulli":
        param["subsample"] = trial.suggest_float("subsample", 0.1, 1, log=True)

    cat_model = cat.CatBoostClassifier(**param, task_type='CPU')
    # cat_model = cat.CatBoostClassifier(**param, task_type='GPU', devices="0:1")
    pruning_callback = CatBoostPruningCallback(trial, "Accuracy")
    cat_model.fit(
        X,y,eval_set=[(X_valid, y_valid)],
        verbose=0,early_stopping_rounds=100,cat_features=cf,
        callbacks=[pruning_callback],
    )

    pruning_callback.check_pruned()
    preds = cat_model.predict(X_valid)
    pred_labels = np.rint(preds)
    accuracy = accuracy_score(y_valid, pred_labels)

    return accuracy


if __name__ == "__main__":
    study = optuna.create_study(
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=5), 
        direction="maximize"
    )
    study.optimize(objective, n_trials=100, timeout=600)

    print("Number of finished trials: {}".format(len(study.trials)))
    print("Best trial:")
    trial = study.best_trial

    print("  Value: {}".format(trial.value))
    print("  Params: ")
    for key, value in trial.params.items():
        print("    {}: {}".format(key, value))

In [ ]:
bast_params = {
    "objective": "Logloss",
    'bootstrap_type': 'MVS',
    'boosting_type': 'Ordered',
    'nan_mode': 'Min',
    'eval_metric':True,
    'use_best_model':True,
    'learning_rate': 0.11363774930060457,
    'colsample_bylevel': 0.7739277478862429,
    'l2_leaf_reg': 0.0021356087830571918,
    'scale_pos_weight': 1.4103615415557027,
    'depth': 11,
    'min_data_in_leaf': 6,
    'max_ctr_complexity': 4,
    'one_hot_max_size': 2,
    "eval_metric": "AUC",
    "random_seed": 42,
    "task_type": "CPU",
    #"task_type": "GPU",
    #"devices": "0:1"
}

In [ ]:
model = cat.CatBoostClassifier(**bast_params, iterations=1000)
model.fit(X=X_train, y=y_train, cat_features=cf, eval_set=[(X_valid, y_valid)], verbose=100, use_best_model=True, plot=False)

# 模型分析

* 模型树可视化
* 特征可解释

In [ ]:
model.plot_tree(tree_idx=1, pool=data_test)

In [ ]:
feature_importance[0]

In [ ]:
feature_importance = pd.DataFrame(model.get_feature_importance(), index=np.array(model.feature_names_)).sort_values(by=[0],ascending=False)
sns.barplot(data=feature_importance, y=feature_importance.index, x=feature_importance[0], orient='h', )

# 模型测试

* 模型预测
* 混淆矩阵
* 测试可视化

## 模型预测

In [ ]:
y_score = model.predict_proba(X_test)
y_infer = model.predict(X_test)

混淆矩阵：对角线比率为准确率、（右下角为原点）第一格和第一行为精准率、（右下角为原点）第一格和第一列为覆盖率（或灵敏度）、第二格和第二列比值特异度（假阳率）

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_infer, labels=[0,1])
display = ConfusionMatrixDisplay(cm)

display.plot(
    include_values=True,
    cmap='viridis',
    ax=None,
    xticks_rotation='horizontal',
    values_format='d'
)

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score,accuracy_score
from sklearn.metrics import precision_score,f1_score

# micro：多分类　　
# weighted：不均衡数量的类来说，计算二分类metrics的平均
# macro：计算二分类metrics的均值，为每个类给出相同权重的分值。
precision = precision_score(y_test, y_infer, average='weighted')
recall = recall_score(y_test, y_infer, average='weighted')
f1_score = f1_score(y_test, y_infer, average='weighted')
accuracy_score = accuracy_score(y_test, y_infer)

print("Precision_score:",precision)
print("Recall_score:",recall)
# F1-Score = 2* 精确分数 * 召回分数/ （精确分数 + 召回分数/）
print("F1_score:",f1_score)
print("Accuracy_score:",accuracy_score)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score, PrecisionRecallDisplay

precision, recall, _ = precision_recall_curve(y_test, y_score[:,1])
pr_display = PrecisionRecallDisplay(
    precision, recall,
    average_precision=average_precision_score(y_test, y_score[:,1]),
    pos_label='1', estimator_name='CC Binary Classifier'
    )
pr_display.plot()

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score,auc,RocCurveDisplay

fpr, tpr, _ = roc_curve(y_test, y_score[:,1])
roc_display = RocCurveDisplay(fpr=fpr,tpr=tpr,roc_auc=auc(fpr, tpr),estimator_name='CC Binary Classifier')
roc_display.plot()

# 提测成果

In [ ]:
# 查看提测数据标准
sample_submission_df = pd.read_csv('/kaggle/input/spaceship-titanic/sample_submission.csv')
#sample_submission_df['Transported'] = n_predictions
sample_submission_df.head()

In [ ]:
(predictions > 0.5).astype(bool).squeeze()

In [ ]:
test_df = pd.read_csv('/kaggle/input/spaceship-titanic/test.csv')
submission_id = test_df.PassengerId

test_df[["Deck", "Cabin_num", "Side"]] = test_df["Cabin"].str.split("/", expand=True)
test_df = test_df.drop('Cabin', axis=1)

test_df['VIP'] = test_df['VIP'].replace({True: 1, False: 0})
test_df['CryoSleep'] = test_df['CryoSleep'].replace({True: 1, False: 0})

cf = ['HomePlanet','Destination','Deck','Side']
test_df.replace({None: np.nan}, inplace=True)
test_df[cf]=dataset_df[cf].replace({np.nan: 'null'})

data_test = cat.Pool(test_df.drop(['PassengerId', 'Name'], axis=1, inplace=False), cat_features=cf)

predictions = model.predict(data_test)
n_predictions = (predictions > 0.5).astype(bool)
output = pd.DataFrame({'PassengerId': submission_id,
                       'Transported': n_predictions.squeeze()})

output.head()

In [ ]:
output.to_csv('/kaggle/working/submission.csv', index=False)